# GRU Hyperparameter Tuning

This notebook compares hyperparameters for the two-layer GRU
using frozen fastText embeddings.

Experiments:
1. Learning rate
2. Dropout
3. Hidden dimension
4. Multi-seed verification

All experiments use training and validation data only.
The test set remains untouched.

Hardware: NVIDIA RTX 4050 (CUDA)
Tracking: Weights & Biases

## Setup and Model Initialization

We load the ART training and validation datasets and reuse the frozen fastText embeddings from Notebook 01.

The GRU is initialized with:
- 2 GRU layers
- Hidden dimension: 128
- Dropout: 0.2
- Batch size: 32
- Random seed: 42

CUDA availability, embedding freezing, and output dimensions are verified before tuning.

In [1]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import wandb

from datasets import load_dataset

# Locate repository root
repo_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "project1" / "artifacts" / "fasttext_art.pt").exists()
    ),
    None
)

assert repo_root is not None, "Project root not found"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from project1.src.models import EmbeddingGRU
from project1.src.data_loader import create_dataloader
from project1.src.training import train_one_epoch, evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# GPU is required for this tuning notebook
assert torch.cuda.is_available(), "CUDA is not available"

device = torch.device("cuda")

print("=== GPU ===")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# Load train and validation data
train_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="train"
)

val_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="validation"
)

# Load verified fastText artifact
artifact_path = (
    repo_root / "project1" / "artifacts" / "fasttext_art.pt"
)

saved = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True
)

vocab = saved["vocab"]
embedding_weights = saved["embedding_weights"]

assert embedding_weights.shape == (15038, 300)
assert vocab["[PAD]"] == 0

# Fixed settings for initial tuning
config = {
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "batch_size": 32,
    "max_epochs": 60,
    "patience": 15,
    "seed": seed
}

# Existing preprocessing and DataLoaders
train_loader = create_dataloader(
    train_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=True
)

val_loader = create_dataloader(
    val_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=False
)

# Model factory: fresh model for every experiment
def create_gru_model(hidden_dim=128, dropout=0.2):

    embedding_layer = nn.Embedding.from_pretrained(
        embedding_weights.detach().cpu().clone(),
        freeze=True,
        padding_idx=vocab["[PAD]"]
    )

    return EmbeddingGRU(
        embedding_layer=embedding_layer,
        pad_id=vocab["[PAD]"],
        hidden_dim=hidden_dim,
        num_layers=2,
        dropout=dropout
    ).to(device)


# Quick independent GPU check
test_model = create_gru_model()

batch = next(iter(val_loader))
test_model.eval()

with torch.inference_mode():
    logits = test_model(
        batch["input_ids"].to(device),
        lengths=batch["lengths"]
    )

assert logits.shape == (batch["labels"].size(0), 2)
assert logits.device.type == "cuda"
assert not test_model.embedding.weight.requires_grad

print("\n=== TUNING SETUP ===")
print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Vocabulary size:", len(vocab))
print("GRU output shape:", tuple(logits.shape))
print("Output device:", logits.device)
print("Model initialization: PASS")
print("Ready for GPU tuning: PASS")

del test_model, logits


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== GPU ===
PyTorch: 2.14.1+cu130
CUDA: 13.0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

=== TUNING SETUP ===
Training samples: 16801
Validation samples: 1000
Vocabulary size: 15038
GRU output shape: (32, 2)
Output device: cuda:0
Model initialization: PASS
Ready for GPU tuning: PASS


## Learning Rate Comparison

We compare three learning rates (0.0001, 0.0003, 0.001) while keeping the GRU architecture and other hyperparameters fixed.

Each experiment uses a freshly initialized model, early stopping, and W&B tracking. The best checkpoint is selected based on validation accuracy.

In [2]:
#don't run
learning_rates = [0.0001, 0.0003, 0.001]

checkpoint_dir = (
    repo_root / "project1" / "artifacts"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

lr_results = []

for lr in learning_rates:

    print(f"\n{'=' * 60}")
    print(f"GRU LEARNING RATE EXPERIMENT: {lr}")
    print(f"{'=' * 60}")

    # 1. Reproducibility
    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])
    torch.cuda.manual_seed_all(config["seed"])

    # 2. Fresh model and optimizer
    experiment_model = create_gru_model(
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"]
    )

    experiment_optimizer = torch.optim.Adam(
        [
            p for p in experiment_model.parameters()
            if p.requires_grad
        ],
        lr=lr,
        weight_decay=0.0
    )

    experiment_criterion = nn.CrossEntropyLoss()

    # Fresh loader for each run
    experiment_train_loader = create_dataloader(
        train_data,
        vocab,
        batch_size=config["batch_size"],
        shuffle=True
    )

    # 3. W&B tracking
    run_name = (
        f"gru_h128_l2_lr{lr}_bs32_"
        f"drop0p2_seed42_cuda"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="gru-learning-rate",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            **config,
            "learning_rate": lr,
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "embedding": "fasttext-wiki-news-subwords-300",
            "embedding_frozen": True,
            "preprocessing": "v2",
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    # 4. Tracking best results
    best_accuracy = -1.0
    loss_at_best_accuracy = float("inf")
    lowest_val_loss = float("inf")

    best_epoch = 0
    no_improvement = 0
    last_epoch = 0

    torch.cuda.reset_peak_memory_stats()

    # Reset training RNG consistently across runs
    random.seed(config["seed"] + 1)
    np.random.seed(config["seed"] + 1)
    torch.manual_seed(config["seed"] + 1)
    torch.cuda.manual_seed_all(config["seed"] + 1)

    try:
        # 5. Training
        for epoch in range(
            1, config["max_epochs"] + 1
        ):
            last_epoch = epoch

            train_metrics = train_one_epoch(
                experiment_model,
                experiment_train_loader,
                experiment_optimizer,
                experiment_criterion,
                device
            )

            val_metrics = evaluate(
                experiment_model,
                val_loader,
                experiment_criterion,
                device
            )

            # W&B logging
            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            lowest_val_loss = min(
                lowest_val_loss,
                val_metrics["loss"]
            )

            # 6. Best checkpoint by validation accuracy
            # Validation loss breaks accuracy ties
            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] <
                    loss_at_best_accuracy
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                loss_at_best_accuracy = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )

            else:
                no_improvement += 1

            print(
                f"Epoch {epoch:02d} | "
                f"Train: {train_metrics['accuracy']:.2%} | "
                f"Val: {val_metrics['accuracy']:.2%} | "
                f"Val loss: {val_metrics['loss']:.4f}"
            )

            # 7. Early stopping
            if no_improvement >= config["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # 8. Restore best checkpoint
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        restored_metrics = evaluate(
            experiment_model,
            val_loader,
            experiment_criterion,
            device
        )

        assert abs(
            restored_metrics["accuracy"] - best_accuracy
        ) < 1e-6

        assert abs(
            restored_metrics["loss"] -
            loss_at_best_accuracy
        ) < 1e-5

        # 9. Store final results
        peak_memory_mb = (
            torch.cuda.max_memory_allocated()
            / 1024**2
        )

        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = (
            loss_at_best_accuracy
        )
        run.summary["lowest_val_loss"] = lowest_val_loss
        run.summary["epochs_trained"] = last_epoch
        run.summary["peak_gpu_memory_mb"] = peak_memory_mb

        lr_results.append({
            "learning_rate": lr,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "loss_at_best": loss_at_best_accuracy,
            "lowest_val_loss": lowest_val_loss,
            "epochs_trained": last_epoch,
            "peak_gpu_memory_mb": peak_memory_mb,
            "checkpoint": str(checkpoint_path)
        })

        print("\n=== EXPERIMENT RESULT ===")
        print(f"Learning rate: {lr}")
        print(f"Best accuracy: {best_accuracy:.2%}")
        print(f"Best epoch: {best_epoch}")
        print(
            f"Loss at best accuracy: "
            f"{loss_at_best_accuracy:.4f}"
        )
        print(f"Lowest val loss: {lowest_val_loss:.4f}")
        print(f"Peak GPU memory: {peak_memory_mb:.1f} MB")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()

    del experiment_model, experiment_optimizer
    torch.cuda.empty_cache()


# ============================================================
# Final Comparison
# ============================================================

print("\n=== GRU LEARNING RATE COMPARISON ===")

lr_results.sort(
    key=lambda result: (
        -result["best_accuracy"],
        result["loss_at_best"]
    )
)

for result in lr_results:
    print(
        f"LR: {result['learning_rate']:<7} | "
        f"Best val acc: {result['best_accuracy']:.2%} | "
        f"Epoch: {result['best_epoch']:02d} | "
        f"Loss at best: {result['loss_at_best']:.4f} | "
        f"Lowest loss: {result['lowest_val_loss']:.4f}"
    )

print("\nBest learning rate:", lr_results[0]["learning_rate"])

# Reference: original GRU baseline (68.10% validation accuracy)
baseline_accuracy = 0.681

improvement = (
    lr_results[0]["best_accuracy"] - baseline_accuracy
) * 100

print(
    f"Difference from original GRU baseline: "
    f"{improvement:+.2f} percentage points"
)



GRU LEARNING RATE EXPERIMENT: 0.0001


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 50.21% | Val: 51.60% | Val loss: 0.6930
Epoch 02 | Train: 51.75% | Val: 52.60% | Val loss: 0.6927
Epoch 03 | Train: 53.81% | Val: 54.10% | Val loss: 0.6914
Epoch 04 | Train: 58.33% | Val: 58.80% | Val loss: 0.6659
Epoch 05 | Train: 62.13% | Val: 59.10% | Val loss: 0.6569
Epoch 06 | Train: 63.78% | Val: 61.90% | Val loss: 0.6492
Epoch 07 | Train: 64.76% | Val: 62.30% | Val loss: 0.6415
Epoch 08 | Train: 65.84% | Val: 62.30% | Val loss: 0.6383
Epoch 09 | Train: 66.75% | Val: 64.80% | Val loss: 0.6310
Epoch 10 | Train: 67.19% | Val: 64.10% | Val loss: 0.6344
Epoch 11 | Train: 67.87% | Val: 63.20% | Val loss: 0.6292
Epoch 12 | Train: 68.17% | Val: 63.80% | Val loss: 0.6238
Epoch 13 | Train: 68.76% | Val: 63.10% | Val loss: 0.6280
Epoch 14 | Train: 68.94% | Val: 64.30% | Val loss: 0.6230
Epoch 15 | Train: 69.28% | Val: 62.70% | Val loss: 0.6399
Epoch 16 | Train: 69.39% | Val: 63.80% | Val loss: 0.6254
Epoch 17 | Train: 70.03% | Val: 64.40% | Val loss: 0.6242
Epoch 18 | Tra

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
train/accuracy,▁▂▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████
train/loss,███▇▇▆▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁
val/accuracy,▁▁▄▆▆▇▆▇▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██▇█▇█▇▇▇▇██
val/loss,██▅▄▃▂▂▂▁▂▂▁▂▁▁▂▁▂▁▂▁▁▂▃▂▄▃▂▃▂▄▄▄▅▄▄▅▇▆▆
best_epoch,50
best_val_accuracy,0.668
epoch,60
epochs_trained,60
lowest_val_loss,0.61759
peak_gpu_memory_mb,58.84033



GRU LEARNING RATE EXPERIMENT: 0.0003


Epoch 01 | Train: 50.66% | Val: 50.70% | Val loss: 0.6928
Epoch 02 | Train: 54.94% | Val: 58.40% | Val loss: 0.6677
Epoch 03 | Train: 62.56% | Val: 61.00% | Val loss: 0.6407
Epoch 04 | Train: 65.78% | Val: 63.80% | Val loss: 0.6253
Epoch 05 | Train: 67.23% | Val: 63.10% | Val loss: 0.6192
Epoch 06 | Train: 67.75% | Val: 63.00% | Val loss: 0.6284
Epoch 07 | Train: 68.81% | Val: 66.50% | Val loss: 0.6134
Epoch 08 | Train: 68.85% | Val: 65.40% | Val loss: 0.6097
Epoch 09 | Train: 69.67% | Val: 65.50% | Val loss: 0.6051
Epoch 10 | Train: 70.59% | Val: 64.60% | Val loss: 0.6259
Epoch 11 | Train: 71.26% | Val: 65.20% | Val loss: 0.6204
Epoch 12 | Train: 72.07% | Val: 65.10% | Val loss: 0.6176
Epoch 13 | Train: 72.38% | Val: 65.50% | Val loss: 0.6119
Epoch 14 | Train: 72.45% | Val: 65.80% | Val loss: 0.6151
Epoch 15 | Train: 73.14% | Val: 65.50% | Val loss: 0.6296
Epoch 16 | Train: 73.68% | Val: 65.50% | Val loss: 0.6175
Epoch 17 | Train: 74.48% | Val: 65.20% | Val loss: 0.6359
Epoch 18 | Tra

epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train/accuracy,▁▂▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
train/loss,██▆▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁
val/accuracy,▁▄▆▇▆▆███▇▇▇████▇▇▇██▇
val/loss,█▆▄▃▂▃▂▁▁▃▂▂▂▂▃▂▃▂▅▃▄▄
best_epoch,7
best_val_accuracy,0.665
epoch,22
epochs_trained,22
lowest_val_loss,0.60512
peak_gpu_memory_mb,58.90967



GRU LEARNING RATE EXPERIMENT: 0.001


Epoch 01 | Train: 52.72% | Val: 60.60% | Val loss: 0.6640
Epoch 02 | Train: 63.70% | Val: 64.30% | Val loss: 0.6212
Epoch 03 | Train: 66.46% | Val: 64.60% | Val loss: 0.6233
Epoch 04 | Train: 68.34% | Val: 65.10% | Val loss: 0.6174
Epoch 05 | Train: 69.28% | Val: 65.80% | Val loss: 0.5992
Epoch 06 | Train: 70.65% | Val: 65.70% | Val loss: 0.6071
Epoch 07 | Train: 72.04% | Val: 66.30% | Val loss: 0.6098
Epoch 08 | Train: 73.32% | Val: 65.00% | Val loss: 0.6028
Epoch 09 | Train: 74.98% | Val: 67.10% | Val loss: 0.6032
Epoch 10 | Train: 76.53% | Val: 64.70% | Val loss: 0.6243
Epoch 11 | Train: 78.14% | Val: 66.00% | Val loss: 0.6696
Epoch 12 | Train: 80.28% | Val: 63.30% | Val loss: 0.7025
Epoch 13 | Train: 82.17% | Val: 62.70% | Val loss: 0.7575
Epoch 14 | Train: 84.26% | Val: 63.20% | Val loss: 0.8209
Epoch 15 | Train: 86.52% | Val: 64.40% | Val loss: 0.9487
Epoch 16 | Train: 88.10% | Val: 63.70% | Val loss: 0.9014
Epoch 17 | Train: 90.26% | Val: 63.80% | Val loss: 1.0714
Epoch 18 | Tra

epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
train/accuracy,▁▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇█████
train/loss,█▇▇▇▇▆▆▆▆▆▅▅▄▄▄▃▃▂▂▂▁▁▁▁
val/accuracy,▁▅▅▆▇▆▇▆█▅▇▄▃▄▅▄▄▃▆▄▄▄▅▄
val/loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▄▄▄▅▇▇██
best_epoch,9
best_val_accuracy,0.671
epoch,24
epochs_trained,24
lowest_val_loss,0.59917
peak_gpu_memory_mb,58.90967



=== GRU LEARNING RATE COMPARISON ===
LR: 0.001   | Best val acc: 67.10% | Epoch: 09 | Loss at best: 0.6032 | Lowest loss: 0.5992
LR: 0.0001  | Best val acc: 66.80% | Epoch: 50 | Loss at best: 0.6556 | Lowest loss: 0.6176
LR: 0.0003  | Best val acc: 66.50% | Epoch: 07 | Loss at best: 0.6134 | Lowest loss: 0.6051

Best learning rate: 0.001
Difference from original GRU baseline: -1.00 percentage points


### Results

| Learning rate | Best validation accuracy | Best epoch |
|---|---:|---:|
| 0.0001 | 66.80% | 50 |
| 0.0003 | 66.50% | 7 |
| 0.001 | **67.10%** | 9 |

Learning rate **0.001** achieved the highest validation accuracy.

However, training accuracy increased substantially while validation performance declined, indicating overfitting.

## Dropout Comparison

We compare dropout rates of 0.0, 0.2, and 0.4 to investigate their effect on GRU generalization.

Fixed settings:
- Learning rate: 0.001
- Hidden dimension: 128
- GRU layers: 2
- Random seed: 42

All experiments use early stopping and W&B tracking.

In [ ]:
#don't run
dropout_values = [0.0, 0.2, 0.4]
selected_lr = 0.001

# Define checkpoint directory independently
checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

dropout_results = []

for dropout in dropout_values:

    print(f"\n{'=' * 60}")
    print(f"GRU DROPOUT EXPERIMENT: {dropout}")
    print(f"{'=' * 60}")

    # 1. Reproducibility
    seed = config["seed"]

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 2. Fresh GRU
    experiment_model = create_gru_model(
        hidden_dim=config["hidden_dim"],
        dropout=dropout
    )

    optimizer = torch.optim.Adam(
        [p for p in experiment_model.parameters()
         if p.requires_grad],
        lr=selected_lr,
        weight_decay=0.0
    )

    criterion = nn.CrossEntropyLoss()

    experiment_train_loader = create_dataloader(
        train_data,
        vocab,
        batch_size=config["batch_size"],
        shuffle=True
    )

    # 3. W&B
    dropout_name = str(dropout).replace(".", "p")

    run_name = (
        f"gru_h128_l2_lr0p001_bs32_"
        f"drop{dropout_name}_seed42_cuda"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="gru-dropout",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            **config,
            "learning_rate": selected_lr,
            "dropout": dropout,
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "embedding_frozen": True,
            "preprocessing": "v2",
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    best_accuracy = -1.0
    loss_at_best = float("inf")
    lowest_loss = float("inf")
    best_epoch = 0
    no_improvement = 0
    last_epoch = 0

    torch.cuda.reset_peak_memory_stats()

    # Match training seed setup from LR experiment
    random.seed(seed + 1)
    np.random.seed(seed + 1)
    torch.manual_seed(seed + 1)
    torch.cuda.manual_seed_all(seed + 1)

    try:
        # 4. Training
        for epoch in range(
            1, config["max_epochs"] + 1
        ):
            last_epoch = epoch

            train_metrics = train_one_epoch(
                experiment_model,
                experiment_train_loader,
                optimizer,
                criterion,
                device
            )

            val_metrics = evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            # W&B metrics
            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            lowest_loss = min(
                lowest_loss,
                val_metrics["loss"]
            )

            # Best checkpoint: accuracy, then loss
            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < loss_at_best
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                loss_at_best = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )
            else:
                no_improvement += 1

            print(
                f"Epoch {epoch:02d} | "
                f"Train: {train_metrics['accuracy']:.2%} | "
                f"Val: {val_metrics['accuracy']:.2%} | "
                f"Val loss: {val_metrics['loss']:.4f}"
            )

            # Early stopping
            if no_improvement >= config["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # 5. Restore checkpoint
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        restored = evaluate(
            experiment_model,
            val_loader,
            criterion,
            device
        )

        assert abs(
            restored["accuracy"] - best_accuracy
        ) < 1e-6

        assert abs(
            restored["loss"] - loss_at_best
        ) < 1e-5

        # 6. Save experiment summary
        peak_memory_mb = (
            torch.cuda.max_memory_allocated()
            / 1024**2
        )

        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = loss_at_best
        run.summary["lowest_val_loss"] = lowest_loss
        run.summary["epochs_trained"] = last_epoch
        run.summary["peak_gpu_memory_mb"] = peak_memory_mb

        dropout_results.append({
            "dropout": dropout,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "loss_at_best": loss_at_best,
            "lowest_val_loss": lowest_loss,
            "epochs_trained": last_epoch,
            "checkpoint": str(checkpoint_path)
        })

        print("\n=== EXPERIMENT RESULT ===")
        print(f"Dropout: {dropout}")
        print(f"Best validation accuracy: {best_accuracy:.2%}")
        print(f"Best epoch: {best_epoch}")
        print(f"Loss at best accuracy: {loss_at_best:.4f}")
        print(f"Lowest validation loss: {lowest_loss:.4f}")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()

    del experiment_model, optimizer
    torch.cuda.empty_cache()


# ============================================================
# Final Dropout Comparison
# ============================================================

print("\n=== GRU DROPOUT COMPARISON ===")

dropout_results.sort(
    key=lambda r: (
        -r["best_accuracy"],
        r["loss_at_best"]
    )
)

for result in dropout_results:
    print(
        f"Dropout: {result['dropout']:.1f} | "
        f"Best val acc: {result['best_accuracy']:.2%} | "
        f"Epoch: {result['best_epoch']:02d} | "
        f"Loss at best: {result['loss_at_best']:.4f} | "
        f"Lowest loss: {result['lowest_val_loss']:.4f}"
    )

print("\nBest dropout:", dropout_results[0]["dropout"])


### Dropout Results

| Dropout | Best validation accuracy | Best epoch |
|---|---:|---:|
| 0.0 | **68.50%** | 9 |
| 0.2 | 67.10% | 9 |
| 0.4 | 66.40% | 10 |

Dropout 0.0 achieved the highest validation accuracy.

Higher dropout rates did not improve performance. All configurations showed overfitting during later epochs.

**Selected dropout: 0.0**

## Hidden Dimension Comparison

We compare GRU hidden dimensions of 64, 128, and 256 to investigate the effect of model capacity.

Fixed settings:
- GRU layers: 2
- Learning rate: 0.001
- Dropout: 0.0
- Random seed: 42

Each configuration is trained independently and tracked using W&B.

In [2]:
#don't run

hidden_dimensions = [64, 128, 256]

selected_lr = 0.001
selected_dropout = 0.0
seed = config["seed"]

# Independent checkpoint directory
checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

hidden_dim_results = []

for hidden_dim in hidden_dimensions:

    print(f"\n{'=' * 60}")
    print(f"GRU HIDDEN DIMENSION: {hidden_dim}")
    print(f"{'=' * 60}")

    # 1. Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 2. Fresh model
    experiment_model = create_gru_model(
        hidden_dim=hidden_dim,
        dropout=selected_dropout
    )

    optimizer = torch.optim.Adam(
        [
            p for p in experiment_model.parameters()
            if p.requires_grad
        ],
        lr=selected_lr,
        weight_decay=0.0
    )

    criterion = nn.CrossEntropyLoss()

    experiment_train_loader = create_dataloader(
        train_data,
        vocab,
        batch_size=config["batch_size"],
        shuffle=True
    )

    trainable_params = sum(
        p.numel()
        for p in experiment_model.parameters()
        if p.requires_grad
    )

    print("Trainable parameters:", trainable_params)

    # 3. W&B tracking
    run_name = (
        f"gru_h{hidden_dim}_l2_lr0p001_"
        f"bs32_drop0p0_seed42_cuda"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="gru-hidden-dimension",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            **config,
            "hidden_dim": hidden_dim,
            "learning_rate": selected_lr,
            "dropout": selected_dropout,
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "trainable_parameters": trainable_params,
            "embedding_frozen": True,
            "preprocessing": "v2",
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    # 4. Tracking
    best_accuracy = -1.0
    loss_at_best = float("inf")
    lowest_loss = float("inf")
    best_epoch = 0
    no_improvement = 0
    last_epoch = 0

    torch.cuda.reset_peak_memory_stats()

    # Match RNG setup from previous experiments
    random.seed(seed + 1)
    np.random.seed(seed + 1)
    torch.manual_seed(seed + 1)
    torch.cuda.manual_seed_all(seed + 1)

    try:
        # 5. Training
        for epoch in range(
            1, config["max_epochs"] + 1
        ):
            last_epoch = epoch

            train_metrics = train_one_epoch(
                experiment_model,
                experiment_train_loader,
                optimizer,
                criterion,
                device
            )

            val_metrics = evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            lowest_loss = min(
                lowest_loss,
                val_metrics["loss"]
            )

            # Best accuracy; loss breaks ties
            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < loss_at_best
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                loss_at_best = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )
            else:
                no_improvement += 1

            if (
                epoch == 1
                or epoch % 5 == 0
                or improved
            ):
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Val: {val_metrics['accuracy']:.2%} | "
                    f"Val loss: {val_metrics['loss']:.4f}"
                )

            if no_improvement >= config["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # 6. Restore best checkpoint
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        restored = evaluate(
            experiment_model,
            val_loader,
            criterion,
            device
        )

        assert abs(
            restored["accuracy"] - best_accuracy
        ) < 1e-6

        assert abs(
            restored["loss"] - loss_at_best
        ) < 1e-5

        peak_memory_mb = (
            torch.cuda.max_memory_allocated()
            / 1024**2
        )

        # 7. Save results
        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = loss_at_best
        run.summary["lowest_val_loss"] = lowest_loss
        run.summary["epochs_trained"] = last_epoch
        run.summary["peak_gpu_memory_mb"] = peak_memory_mb

        hidden_dim_results.append({
            "hidden_dim": hidden_dim,
            "trainable_parameters": trainable_params,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "loss_at_best": loss_at_best,
            "lowest_val_loss": lowest_loss,
            "epochs_trained": last_epoch,
            "peak_gpu_memory_mb": peak_memory_mb,
            "checkpoint": str(checkpoint_path)
        })

        print("\n=== EXPERIMENT RESULT ===")
        print("Hidden dimension:", hidden_dim)
        print(f"Best validation accuracy: {best_accuracy:.2%}")
        print("Best epoch:", best_epoch)
        print(f"Loss at best: {loss_at_best:.4f}")
        print(f"Lowest validation loss: {lowest_loss:.4f}")
        print(f"Peak GPU memory: {peak_memory_mb:.1f} MB")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()

    del experiment_model, optimizer
    torch.cuda.empty_cache()


# ============================================================
# Final Hidden Dimension Comparison
# ============================================================

print("\n=== GRU HIDDEN DIMENSION COMPARISON ===")

hidden_dim_results.sort(
    key=lambda r: (
        -r["best_accuracy"],
        r["loss_at_best"]
    )
)

for result in hidden_dim_results:
    print(
        f"Hidden: {result['hidden_dim']:<3} | "
        f"Params: {result['trainable_parameters']:<8} | "
        f"Best val acc: {result['best_accuracy']:.2%} | "
        f"Epoch: {result['best_epoch']:02d} | "
        f"Lowest loss: {result['lowest_val_loss']:.4f}"
    )

print("\nBest hidden dimension:",
      hidden_dim_results[0]["hidden_dim"])



GRU HIDDEN DIMENSION: 64
Trainable parameters: 99522


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 52.38% | Val: 60.30% | Val loss: 0.6637
Epoch 02 | Train: 63.45% | Val: 64.60% | Val loss: 0.6165
Epoch 03 | Train: 66.68% | Val: 65.00% | Val loss: 0.6177
Epoch 04 | Train: 68.35% | Val: 65.10% | Val loss: 0.6106
Epoch 05 | Train: 69.56% | Val: 65.20% | Val loss: 0.5993
Epoch 06 | Train: 70.73% | Val: 65.90% | Val loss: 0.6076
Epoch 07 | Train: 71.91% | Val: 66.70% | Val loss: 0.6041
Epoch 09 | Train: 73.83% | Val: 67.20% | Val loss: 0.5961
Epoch 10 | Train: 75.03% | Val: 68.00% | Val loss: 0.6085
Epoch 15 | Train: 81.79% | Val: 66.40% | Val loss: 0.6612
Epoch 20 | Train: 88.85% | Val: 65.50% | Val loss: 0.8221
Epoch 25 | Train: 94.39% | Val: 63.80% | Val loss: 1.4582
Early stopping at epoch 25

=== EXPERIMENT RESULT ===
Hidden dimension: 64
Best validation accuracy: 68.00%
Best epoch: 10
Loss at best: 0.6085
Lowest validation loss: 0.5961
Peak GPU memory: 54.2 MB
Checkpoint verification: PASS


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/accuracy,▁▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇████
train/loss,█▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▂▂▂▁▁
val/accuracy,▁▅▅▅▅▆▇▇▇██▇▇█▇▆▆▆▅▆▆▅▅▅▄
val/loss,▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▃▃▃▃▅▅▆▆█
best_epoch,10
best_val_accuracy,0.68
epoch,25
epochs_trained,25
lowest_val_loss,0.59605
peak_gpu_memory_mb,54.15381



GRU HIDDEN DIMENSION: 128
Trainable parameters: 280962


Epoch 01 | Train: 52.33% | Val: 59.20% | Val loss: 0.6663
Epoch 02 | Train: 62.66% | Val: 62.60% | Val loss: 0.6273
Epoch 03 | Train: 66.38% | Val: 63.80% | Val loss: 0.6291
Epoch 04 | Train: 68.35% | Val: 65.90% | Val loss: 0.6121
Epoch 05 | Train: 69.76% | Val: 66.40% | Val loss: 0.5958
Epoch 08 | Train: 74.22% | Val: 66.50% | Val loss: 0.6031
Epoch 09 | Train: 75.81% | Val: 68.50% | Val loss: 0.5945
Epoch 10 | Train: 77.54% | Val: 67.80% | Val loss: 0.6143
Epoch 15 | Train: 88.19% | Val: 64.70% | Val loss: 0.8426
Epoch 20 | Train: 95.24% | Val: 64.90% | Val loss: 1.5612
Early stopping at epoch 24

=== EXPERIMENT RESULT ===
Hidden dimension: 128
Best validation accuracy: 68.50%
Best epoch: 9
Loss at best: 0.5945
Lowest validation loss: 0.5945
Peak GPU memory: 57.9 MB
Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
train/accuracy,▁▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇██████
train/loss,█▇▇▇▇▆▆▆▆▅▅▅▄▄▃▃▂▂▂▂▁▁▁▁
val/accuracy,▁▄▄▆▆▆▆▆█▇▇▆▆▇▅▇▆▆▇▅▇█▆▆
val/loss,▁▁▁▁▁▁▁▁▁▁▂▂▂▃▂▃▄▄▆▆▇███
best_epoch,9
best_val_accuracy,0.685
epoch,24
epochs_trained,24
lowest_val_loss,0.59448
peak_gpu_memory_mb,57.94678



GRU HIDDEN DIMENSION: 256
Trainable parameters: 889602


Epoch 01 | Train: 53.97% | Val: 58.20% | Val loss: 0.6634
Epoch 02 | Train: 63.79% | Val: 63.60% | Val loss: 0.6221
Epoch 03 | Train: 66.90% | Val: 63.80% | Val loss: 0.6243
Epoch 04 | Train: 68.70% | Val: 65.90% | Val loss: 0.6060
Epoch 05 | Train: 70.20% | Val: 65.80% | Val loss: 0.5971
Epoch 08 | Train: 74.78% | Val: 66.60% | Val loss: 0.6120
Epoch 10 | Train: 79.51% | Val: 64.10% | Val loss: 0.6666
Epoch 15 | Train: 94.13% | Val: 64.90% | Val loss: 1.1729
Epoch 20 | Train: 98.55% | Val: 64.90% | Val loss: 1.9151
Early stopping at epoch 23

=== EXPERIMENT RESULT ===
Hidden dimension: 256
Best validation accuracy: 66.60%
Best epoch: 8
Loss at best: 0.6120
Lowest validation loss: 0.5971
Peak GPU memory: 83.1 MB
Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
train/accuracy,▁▃▃▃▄▄▄▄▅▅▅▆▆▇▇▇███████
train/loss,█▇▇▇▇▇▆▆▆▅▅▄▃▃▂▂▂▁▁▁▁▁▁
val/accuracy,▁▅▆▇▇▇▇██▆▆▆▇▆▇▅▆▆▆▇▆▇▇
val/loss,▁▁▁▁▁▁▁▁▁▁▂▂▃▄▄▅▆▇▇███▇
best_epoch,8
best_val_accuracy,0.666
epoch,23
epochs_trained,23
lowest_val_loss,0.59711
peak_gpu_memory_mb,83.08203



=== GRU HIDDEN DIMENSION COMPARISON ===
Hidden: 128 | Params: 280962   | Best val acc: 68.50% | Epoch: 09 | Lowest loss: 0.5945
Hidden: 64  | Params: 99522    | Best val acc: 68.00% | Epoch: 10 | Lowest loss: 0.5961
Hidden: 256 | Params: 889602   | Best val acc: 66.60% | Epoch: 08 | Lowest loss: 0.5971

Best hidden dimension: 128


### Hidden Dimension Results

| Hidden dimension | Trainable parameters | Best validation accuracy |
|---|---:|---:|
| 64 | 99,522 | 68.00% |
| 128 | 280,962 | **68.50%** |
| 256 | 889,602 | 66.60% |

A hidden dimension of 128 achieved the highest validation accuracy.

Increasing the hidden dimension to 256 increased model complexity but reduced validation performance, suggesting that a larger model does not necessarily generalize better.

**Selected hidden dimension: 128.**

## Multi-Seed Verification

We compare hidden dimensions 64 and 128 across three random seeds (42, 123, 2026) to assess training stability.

Fixed settings:
- Learning rate: 0.001
- Dropout: 0.0
- GRU layers: 2
- Maximum epochs: 60
- Early stopping patience: 15

All experiments are tracked with W&B.

In [2]:
#don't run

import statistics

seeds = [42, 123, 2026]
hidden_dimensions = [64, 128]

selected_lr = 0.001
selected_dropout = 0.0

max_epochs = 60
patience = 15

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

multiseed_results = []

for seed in seeds:
    for hidden_dim in hidden_dimensions:

        print(f"\n{'=' * 60}")
        print(f"GRU | Seed: {seed} | Hidden: {hidden_dim}")
        print(f"{'=' * 60}")

        # 1. Reproducible initialization
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # 2. Fresh model
        experiment_model = create_gru_model(
            hidden_dim=hidden_dim,
            dropout=selected_dropout
        )

        optimizer = torch.optim.Adam(
            [
                p for p in experiment_model.parameters()
                if p.requires_grad
            ],
            lr=selected_lr,
            weight_decay=0.0
        )

        criterion = nn.CrossEntropyLoss()

        experiment_train_loader = create_dataloader(
            train_data,
            vocab,
            batch_size=config["batch_size"],
            shuffle=True
        )

        # 3. W&B run
        run_name = (
            f"gru_multiseed_h{hidden_dim}_"
            f"lr0p001_drop0p0_seed{seed}_cuda"
        )

        run = wandb.init(
            entity="kimbortis-hochschule-luzern",
            project="nlp-art-hs26-project1",
            group="gru-multiseed",
            job_type="validation",
            name=run_name,
            config={
                "architecture": "GRU",
                "hidden_dim": hidden_dim,
                "num_layers": 2,
                "dropout": selected_dropout,
                "learning_rate": selected_lr,
                "batch_size": config["batch_size"],
                "optimizer": "Adam",
                "weight_decay": 0.0,
                "seed": seed,
                "max_epochs": max_epochs,
                "patience": patience,
                "embedding_frozen": True,
                "preprocessing": "v2",
                "device": str(device)
            }
        )

        run.define_metric("epoch")
        run.define_metric("train/*", step_metric="epoch")
        run.define_metric("val/*", step_metric="epoch")

        checkpoint_path = (
            checkpoint_dir / f"{run_name}_{run.id}_best.pt"
        )

        best_accuracy = -1.0
        best_loss = float("inf")
        lowest_loss = float("inf")
        best_epoch = 0
        no_improvement = 0

        # Match the RNG setup of previous CUDA sweeps
        random.seed(seed + 1)
        np.random.seed(seed + 1)
        torch.manual_seed(seed + 1)
        torch.cuda.manual_seed_all(seed + 1)

        try:
            # 4. Training
            for epoch in range(1, max_epochs + 1):

                train_metrics = train_one_epoch(
                    experiment_model,
                    experiment_train_loader,
                    optimizer,
                    criterion,
                    device
                )

                val_metrics = evaluate(
                    experiment_model,
                    val_loader,
                    criterion,
                    device
                )

                run.log({
                    "epoch": epoch,
                    "train/loss": train_metrics["loss"],
                    "train/accuracy": train_metrics["accuracy"],
                    "val/loss": val_metrics["loss"],
                    "val/accuracy": val_metrics["accuracy"]
                })

                lowest_loss = min(
                    lowest_loss,
                    val_metrics["loss"]
                )

                # Best validation accuracy; loss breaks ties
                improved = (
                    val_metrics["accuracy"] > best_accuracy
                    or (
                        val_metrics["accuracy"] == best_accuracy
                        and val_metrics["loss"] < best_loss
                    )
                )

                if improved:
                    best_accuracy = val_metrics["accuracy"]
                    best_loss = val_metrics["loss"]
                    best_epoch = epoch
                    no_improvement = 0

                    torch.save(
                        experiment_model.state_dict(),
                        checkpoint_path
                    )
                else:
                    no_improvement += 1

                if no_improvement >= patience:
                    print(f"Early stopping: epoch {epoch}")
                    break

            # 5. Restore and verify best checkpoint
            experiment_model.load_state_dict(
                torch.load(
                    checkpoint_path,
                    map_location=device,
                    weights_only=True
                )
            )

            restored = evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            assert abs(
                restored["accuracy"] - best_accuracy
            ) < 1e-6

            assert abs(
                restored["loss"] - best_loss
            ) < 1e-5

            # 6. Store results
            run.summary["best_val_accuracy"] = best_accuracy
            run.summary["best_epoch"] = best_epoch
            run.summary["val_loss_at_best_accuracy"] = best_loss
            run.summary["lowest_val_loss"] = lowest_loss
            run.summary["epochs_trained"] = epoch

            multiseed_results.append({
                "seed": seed,
                "hidden_dim": hidden_dim,
                "best_accuracy": best_accuracy,
                "best_loss": best_loss,
                "lowest_val_loss": lowest_loss,
                "best_epoch": best_epoch,
                "checkpoint": str(checkpoint_path)
            })

            print(
                f"Seed {seed} | Hidden {hidden_dim} | "
                f"Accuracy: {best_accuracy:.2%} | "
                f"Best epoch: {best_epoch} | "
                f"Lowest loss: {lowest_loss:.4f}"
            )

            print("Checkpoint verification: PASS")

        finally:
            run.finish()
            del experiment_model, optimizer
            torch.cuda.empty_cache()


# ============================================================
# Multi-Seed Summary
# ============================================================

print("\n=== GRU MULTI-SEED COMPARISON ===")

for hidden_dim in hidden_dimensions:

    scores = [
        r["best_accuracy"]
        for r in multiseed_results
        if r["hidden_dim"] == hidden_dim
    ]

    print(
        f"Hidden {hidden_dim}: "
        f"Mean = {statistics.mean(scores):.2%} | "
        f"Std = {statistics.stdev(scores):.2%} | "
        f"Min = {min(scores):.2%} | "
        f"Max = {max(scores):.2%}"
    )


print("\n=== PAIRED SEED DIFFERENCES ===")

for seed in seeds:

    scores = {
        r["hidden_dim"]: r["best_accuracy"]
        for r in multiseed_results
        if r["seed"] == seed
    }

    difference = 100 * (scores[128] - scores[64])

    print(
        f"Seed {seed}: "
        f"Hidden 128 vs 64 = {difference:+.2f} pp"
    )


print("\n=== MEAN ACCURACY DIFFERENCE ===")

mean_64 = statistics.mean(
    r["best_accuracy"]
    for r in multiseed_results
    if r["hidden_dim"] == 64
)

mean_128 = statistics.mean(
    r["best_accuracy"]
    for r in multiseed_results
    if r["hidden_dim"] == 128
)

print(
    f"128 vs 64: "
    f"{100 * (mean_128 - mean_64):+.2f} pp"
)



GRU | Seed: 42 | Hidden: 64


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Early stopping: epoch 25
Seed 42 | Hidden 64 | Accuracy: 68.00% | Best epoch: 10 | Lowest loss: 0.5961
Checkpoint verification: PASS


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/accuracy,▁▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇████
train/loss,█▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▂▂▂▁▁
val/accuracy,▁▅▅▅▅▆▇▇▇██▇▇█▇▆▆▆▅▆▆▅▅▅▄
val/loss,▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▃▃▃▃▅▅▆▆█
best_epoch,10
best_val_accuracy,0.68
epoch,25
epochs_trained,25
lowest_val_loss,0.59605
train/accuracy,0.94387



GRU | Seed: 42 | Hidden: 128


Early stopping: epoch 24
Seed 42 | Hidden 128 | Accuracy: 68.50% | Best epoch: 9 | Lowest loss: 0.5945
Checkpoint verification: PASS


... (10 outputs left out)


GRU | Seed: 2026 | Hidden: 128


Early stopping: epoch 22
Seed 2026 | Hidden 128 | Accuracy: 67.90% | Best epoch: 7 | Lowest loss: 0.6033
Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train/accuracy,▁▃▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇████
train/loss,██▇▇▇▆▆▆▆▅▅▅▄▄▃▃▂▂▂▁▁▁
val/accuracy,▁▆██▇▇█▆▇▆▇▆▆▇▆▆▆▇█▇▇▆
val/loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▃▄▄▆▆▇█
best_epoch,7
best_val_accuracy,0.679
epoch,22
epochs_trained,22
lowest_val_loss,0.60332
train/accuracy,0.975



=== GRU MULTI-SEED COMPARISON ===
Hidden 64: Mean = 67.33% | Std = 0.76% | Min = 66.50% | Max = 68.00%
Hidden 128: Mean = 68.03% | Std = 0.42% | Min = 67.70% | Max = 68.50%

=== PAIRED SEED DIFFERENCES ===
Seed 42: Hidden 128 vs 64 = +0.50 pp
Seed 123: Hidden 128 vs 64 = +0.20 pp
Seed 2026: Hidden 128 vs 64 = +1.40 pp

=== MEAN ACCURACY DIFFERENCE ===
128 vs 64: +0.70 pp


### Multi-Seed Results

| Hidden dimension | Mean validation accuracy | Standard deviation |
|---|---:|---:|
| 64 | 67.33% | 0.76% |
| 128 | **68.03%** | 0.42% |

Hidden dimension 128 performed better in all three seeds, improving mean validation accuracy by **0.70 percentage points**.

It also showed lower variability across seeds.

**Selected hidden dimension: 128.**

## Validation Diagnostics and Class Bias

We evaluate the selected GRU (hidden dimension 128, seed 42) using a confusion matrix.

We analyze:
- Overall and balanced accuracy
- Recall for H1 and H2
- Prediction distribution

This helps identify possible class or answer-position bias.

In [ ]:
# don't run

# Use our fixed reference seed (42)
selected_run = next(
    r for r in multiseed_results
    if r["seed"] == 42 and r["hidden_dim"] == 128
)

diagnostic_model = create_gru_model(
    hidden_dim=128,
    dropout=0.0
)

diagnostic_model.load_state_dict(
    torch.load(
        selected_run["checkpoint"],
        map_location=device,
        weights_only=True
    )
)

diagnostic_model.eval()

# Rows = true labels; columns = predicted labels
confusion = torch.zeros((2, 2), dtype=torch.long)

with torch.inference_mode():
    for batch in val_loader:

        inputs = batch["input_ids"].to(device)
        lengths = batch["lengths"]
        labels = batch["labels"].cpu().long()

        logits = diagnostic_model(inputs, lengths=lengths)
        predictions = logits.argmax(dim=1).cpu()

        confusion += torch.bincount(
            labels * 2 + predictions,
            minlength=4
        ).reshape(2, 2)

# Metrics
total = confusion.sum().item()
accuracy = confusion.diag().sum().item() / total

class_recall = confusion.diag().float() / (
    confusion.sum(dim=1).clamp(min=1)
)

balanced_accuracy = class_recall.mean().item()

predicted_counts = confusion.sum(dim=0)

print("=== GRU VALIDATION DIAGNOSTICS ===")
print("Validation samples:", total)
print(f"Overall accuracy: {accuracy:.2%}")
print(f"Balanced accuracy: {balanced_accuracy:.2%}")

print("\nConfusion matrix (rows=true, columns=predicted):")
print("               Pred H1  Pred H2")
print(f"True H1         {confusion[0,0]:5d}    {confusion[0,1]:5d}")
print(f"True H2         {confusion[1,0]:5d}    {confusion[1,1]:5d}")

print("\nPer-class recall:")
print(f"H1: {class_recall[0].item():.2%}")
print(f"H2: {class_recall[1].item():.2%}")

print("\nPrediction distribution:")
print(f"Predicted H1: {predicted_counts[0].item() / total:.2%}")
print(f"Predicted H2: {predicted_counts[1].item() / total:.2%}")

assert abs(accuracy - selected_run["best_accuracy"]) < 1e-6

print("\nCheckpoint verification: PASS")


In [2]:
checkpoint_path = (
    repo_root / "project1" / "artifacts" /
    "gru_multiseed_h128_lr0p001_drop0p0_seed42_cuda_hvxtunb8_best.pt"
)

assert checkpoint_path.exists(), (
    f"Checkpoint not found: {checkpoint_path}"
)

diagnostic_model = create_gru_model(
    hidden_dim=128,
    dropout=0.0
)

diagnostic_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

diagnostic_model.eval()

# Verify restored accuracy
restored_metrics = evaluate(
    diagnostic_model,
    val_loader,
    nn.CrossEntropyLoss(),
    device
)

print("=== RESTORED GRU ===")
print("Device:", device)
print(
    f"Validation accuracy: "
    f"{restored_metrics['accuracy']:.2%}"
)

assert abs(
    restored_metrics["accuracy"] - 0.685
) < 1e-6

print("Checkpoint restoration: PASS")
print("\nValidation columns:", val_data.column_names)


=== RESTORED GRU ===
Device: cuda
Validation accuracy: 68.50%
Checkpoint restoration: PASS

Validation columns: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label']


In [3]:

# ============================================================
# GRU Hypothesis-Swapping Diagnostic
# ============================================================

# The original ART labels are 1 (H1) and 2 (H2).
assert set(val_data["label"]) == {1, 2}

# 1. Swap hypotheses and invert the correct answer.
def swap_hypotheses(example):
    return {
        "hypothesis_1": example["hypothesis_2"],
        "hypothesis_2": example["hypothesis_1"],
        "label": 3 - int(example["label"])
    }

swapped_val_data = val_data.map(
    swap_hypotheses,
    load_from_cache_file=False
)

assert (
    swapped_val_data[0]["hypothesis_1"]
    == val_data[0]["hypothesis_2"]
)

assert (
    swapped_val_data[0]["hypothesis_2"]
    == val_data[0]["hypothesis_1"]
)

assert (
    swapped_val_data[0]["label"]
    == 3 - val_data[0]["label"]
)

print("Hypothesis swapping: PASS")

swapped_val_loader = create_dataloader(
    swapped_val_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=False
)

# 2. Get predictions from the same model.
def collect_predictions(model, loader):
    predictions = []
    labels = []

    model.eval()

    with torch.inference_mode():
        for batch in loader:
            logits = model(
                batch["input_ids"].to(device),
                lengths=batch["lengths"]
            )

            predictions.append(
                logits.argmax(dim=1).cpu()
            )

            labels.append(
                batch["labels"].cpu()
            )

    return (
        torch.cat(predictions),
        torch.cat(labels)
    )

original_pred, original_labels = collect_predictions(
    diagnostic_model, val_loader
)

swapped_pred, swapped_labels = collect_predictions(
    diagnostic_model, swapped_val_loader
)

# 3. Verify ground-truth labels were inverted.
assert torch.equal(
    swapped_labels,
    1 - original_labels
)

# 4. Calculate diagnostic metrics.
original_accuracy = (
    original_pred == original_labels
).float().mean().item()

swapped_accuracy = (
    swapped_pred == swapped_labels
).float().mean().item()

# Did the model reverse its prediction after swapping?
flip_consistency = (
    original_pred == (1 - swapped_pred)
).float().mean().item()

both_correct = (
    (original_pred == original_labels)
    & (swapped_pred == swapped_labels)
).float().mean().item()

# 5. Results.
print("=== GRU HYPOTHESIS-SWAPPING DIAGNOSTIC ===")

print(f"Original accuracy: {original_accuracy:.2%}")
print(f"Swapped accuracy: {swapped_accuracy:.2%}")

print(
    f"Accuracy difference: "
    f"{100 * (swapped_accuracy - original_accuracy):+.2f} pp"
)

print(f"Prediction-flip consistency: {flip_consistency:.2%}")
print(f"Correct in both orders: {both_correct:.2%}")

print("\nPrediction distribution:")

print(
    f"Original predicted H2: "
    f"{(original_pred == 1).float().mean().item():.2%}"
)

print(
    f"Swapped predicted H2: "
    f"{(swapped_pred == 1).float().mean().item():.2%}"
)

print("\nHypothesis swap completed: PASS")


Map: 100%|██████████| 1000/1000 [00:00<00:00, 7240.12 examples/s]


Hypothesis swapping: PASS
=== GRU HYPOTHESIS-SWAPPING DIAGNOSTIC ===
Original accuracy: 68.50%
Swapped accuracy: 67.00%
Accuracy difference: -1.50 pp
Prediction-flip consistency: 72.70%
Correct in both orders: 54.10%

Prediction distribution:
Original predicted H2: 53.10%
Swapped predicted H2: 55.20%

Hypothesis swap completed: PASS


## Hypothesis-Swapping Analysis

To investigate answer-position sensitivity, H1 and H2 were
exchanged in every validation example, and the ground-truth
labels were inverted accordingly.

The same trained GRU checkpoint was evaluated without
retraining.

| Metric | Result |
|---|---:|
| Original accuracy | 68.50% |
| Swapped accuracy | 67.00% |
| Accuracy difference | -1.50 pp |
| Prediction-flip consistency | 72.70% |
| Correct in both orders | 54.10% |
| Predicted H2 (original) | 53.10% |
| Predicted H2 (swapped) | 55.20% |

The model maintained similar overall accuracy after swapping
the hypotheses, but prediction-flip consistency was only
72.70%.

This indicates that model decisions are sensitive to the
ordering of the candidate hypotheses.

A possible contributing factor is the sequential input
representation, in which H2 always appears last.

The results do not conclusively establish the cause of this
behavior. A shared hypothesis-scoring architecture could
be investigated in future work.

This diagnostic was performed using validation data only.
The results reveal answer-order sensitivity, motivating the hypothesis-swapping augmentation experiment in the next section.

## Hypothesis-Swapping Data Augmentation

To reduce answer-position bias, H1 and H2 are randomly swapped during training with a probability of 0.5.

When hypotheses are swapped, the label is inverted accordingly.

- Dataset size remains unchanged.
- Original validation data is used for model selection.
- Three seeds (42, 123, 2026) are evaluated.
- Experiments are tracked with W&B.

In [4]:

import random
from torch.utils.data import Dataset

class HypothesisSwapDataset(Dataset):
    """
    Randomly swap H1 and H2 during training.

    swap_probability=0.0: original examples only
    swap_probability=0.5: random swapping
    swap_probability=1.0: always swapped

    The dataset size remains unchanged.
    """

    def __init__(self, dataset, swap_probability=0.5):
        self.dataset = dataset
        self.swap_probability = swap_probability

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = dict(self.dataset[idx])

        if random.random() < self.swap_probability:

            # Swap candidate hypotheses
            example["hypothesis_1"], example["hypothesis_2"] = (
                example["hypothesis_2"],
                example["hypothesis_1"]
            )

            # Invert ART label: 1 <-> 2
            example["label"] = 3 - int(example["label"])

        return example


# ============================================================
# Verification
# ============================================================

# Always-swapped dataset, only for testing
test_swap_dataset = HypothesisSwapDataset(
    train_data,
    swap_probability=1.0
)

original = train_data[0]
swapped = test_swap_dataset[0]

assert swapped["hypothesis_1"] == original["hypothesis_2"]
assert swapped["hypothesis_2"] == original["hypothesis_1"]
assert swapped["label"] == 3 - original["label"]

# Verify the size does not change
assert len(test_swap_dataset) == len(train_data)

print("=== AUGMENTATION VERIFICATION ===")
print("Original H1:", original["hypothesis_1"])
print("Original H2:", original["hypothesis_2"])
print("Original label:", original["label"])

print("\nSwapped H1:", swapped["hypothesis_1"])
print("Swapped H2:", swapped["hypothesis_2"])
print("Swapped label:", swapped["label"])

print("\nHypothesis swapping: PASS")
print("Dataset size unchanged: PASS")


# ============================================================
# Verify compatibility with the existing DataLoader
# ============================================================

augmented_train_data = HypothesisSwapDataset(
    train_data,
    swap_probability=0.5
)

augmented_train_loader = create_dataloader(
    augmented_train_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=True
)

batch = next(iter(augmented_train_loader))

assert batch["input_ids"].shape[0] == config["batch_size"]
assert batch["labels"].shape[0] == config["batch_size"]
assert set(batch["labels"].tolist()).issubset({0, 1})

print("\nAugmented DataLoader: PASS")
print("Batch shape:", tuple(batch["input_ids"].shape))
print("Training examples:", len(augmented_train_data))


=== AUGMENTATION VERIFICATION ===
Original H1: He was never lost.
Original H2: All the people from the zoo looked for him.
Original label: 2

Swapped H1: All the people from the zoo looked for him.
Swapped H2: He was never lost.
Swapped label: 1

Hypothesis swapping: PASS
Dataset size unchanged: PASS

Augmented DataLoader: PASS
Batch shape: (32, 73)
Training examples: 16801


In [5]:

# ============================================================
# Augmented GRU Multi-Seed Training
# ============================================================

import statistics

seeds = [42, 123, 2026]

lr = 0.001
dropout = 0.0
hidden_dim = 128
swap_probability = 0.5
max_epochs = 60
patience = 15

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Existing baseline results from the previous experiment
baseline_scores = {
    42: 0.685,
    123: 0.677,
    2026: 0.679
}

augmentation_results = []

# ------------------------------------------------------------
# Prepare swapped validation data for diagnostic evaluation
# (The training-validation model selection uses ORIGINAL data)
# ------------------------------------------------------------

def swap_validation_example(example):
    return {
        "hypothesis_1": example["hypothesis_2"],
        "hypothesis_2": example["hypothesis_1"],
        "label": 3 - int(example["label"])
    }

swapped_val_data = val_data.map(
    swap_validation_example,
    load_from_cache_file=False
)

swapped_val_loader = create_dataloader(
    swapped_val_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=False
)

def get_predictions(model, loader):
    predictions = []
    labels = []

    model.eval()

    with torch.inference_mode():
        for batch in loader:
            logits = model(
                batch["input_ids"].to(device),
                lengths=batch["lengths"]
            )

            predictions.append(
                logits.argmax(dim=1).cpu()
            )
            labels.append(batch["labels"].cpu())

    return torch.cat(predictions), torch.cat(labels)


# ------------------------------------------------------------
# Train three independent augmented GRUs
# ------------------------------------------------------------

for seed in seeds:

    print(f"\n{'=' * 60}")
    print(f"AUGMENTED GRU | SEED {seed}")
    print(f"{'=' * 60}")

    # 1. Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 2. Fresh model
    experiment_model = create_gru_model(
        hidden_dim=hidden_dim,
        dropout=dropout
    )

    optimizer = torch.optim.Adam(
        [
            p for p in experiment_model.parameters()
            if p.requires_grad
        ],
        lr=lr,
        weight_decay=0.0
    )

    criterion = nn.CrossEntropyLoss()

    # New augmented DataLoader
    augmented_dataset = HypothesisSwapDataset(
        train_data,
        swap_probability=swap_probability
    )

    augmented_loader = create_dataloader(
        augmented_dataset,
        vocab,
        batch_size=config["batch_size"],
        shuffle=True
    )

    # 3. W&B
    run_name = (
        f"gru_aug_swap0p5_h128_lr0p001_"
        f"drop0p0_seed{seed}_cuda"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="gru-hypothesis-augmentation",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            "architecture": "GRU",
            "hidden_dim": hidden_dim,
            "num_layers": 2,
            "learning_rate": lr,
            "dropout": dropout,
            "batch_size": config["batch_size"],
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "swap_probability": swap_probability,
            "seed": seed,
            "max_epochs": max_epochs,
            "patience": patience,
            "embedding_frozen": True,
            "preprocessing": "v2",
            "device": str(device)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    best_accuracy = -1.0
    best_loss = float("inf")
    best_epoch = 0
    no_improvement = 0

    # Match RNG setup of our previous GPU comparisons
    random.seed(seed + 1)
    np.random.seed(seed + 1)
    torch.manual_seed(seed + 1)
    torch.cuda.manual_seed_all(seed + 1)

    try:
        # 4. Training loop
        for epoch in range(1, max_epochs + 1):

            train_metrics = train_one_epoch(
                experiment_model,
                augmented_loader,
                optimizer,
                criterion,
                device
            )

            # IMPORTANT: Always evaluate on original validation
            val_metrics = evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < best_loss
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                best_loss = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )
            else:
                no_improvement += 1

            if epoch == 1 or epoch % 5 == 0 or improved:
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Val: {val_metrics['accuracy']:.2%} | "
                    f"Val loss: {val_metrics['loss']:.4f}"
                )

            if no_improvement >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        # 5. Restore best checkpoint
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        original_pred, original_labels = get_predictions(
            experiment_model, val_loader
        )

        swapped_pred, swapped_labels = get_predictions(
            experiment_model, swapped_val_loader
        )

        assert torch.equal(
            swapped_labels,
            1 - original_labels
        )

        original_accuracy = (
            original_pred == original_labels
        ).float().mean().item()

        swapped_accuracy = (
            swapped_pred == swapped_labels
        ).float().mean().item()

        flip_consistency = (
            original_pred == (1 - swapped_pred)
        ).float().mean().item()

        both_correct = (
            (original_pred == original_labels)
            & (swapped_pred == swapped_labels)
        ).float().mean().item()

        assert abs(
            original_accuracy - best_accuracy
        ) < 1e-6

        # 6. Save results
        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = best_loss
        run.summary["epochs_trained"] = epoch
        run.summary["swapped_accuracy"] = swapped_accuracy
        run.summary["flip_consistency"] = flip_consistency
        run.summary["both_correct"] = both_correct

        augmentation_results.append({
            "seed": seed,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "best_loss": best_loss,
            "swapped_accuracy": swapped_accuracy,
            "flip_consistency": flip_consistency,
            "both_correct": both_correct,
            "checkpoint": str(checkpoint_path)
        })

        print("\n=== AUGMENTED GRU RESULT ===")
        print("Seed:", seed)
        print(f"Original accuracy: {original_accuracy:.2%}")
        print(f"Swapped accuracy: {swapped_accuracy:.2%}")
        print(f"Flip consistency: {flip_consistency:.2%}")
        print(f"Correct in both orders: {both_correct:.2%}")
        print(f"Best epoch: {best_epoch}")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()
        del experiment_model, optimizer
        torch.cuda.empty_cache()


# ============================================================
# Final Comparison
# ============================================================

print("\n=== AUGMENTATION MULTI-SEED COMPARISON ===")

augmented_scores = []

for result in augmentation_results:
    seed = result["seed"]
    augmented_acc = result["best_accuracy"]
    baseline_acc = baseline_scores[seed]

    augmented_scores.append(augmented_acc)

    print(
        f"Seed {seed}: "
        f"Baseline={baseline_acc:.2%} | "
        f"Augmented={augmented_acc:.2%} | "
        f"Difference={100 * (augmented_acc - baseline_acc):+.2f} pp"
    )

baseline_mean = statistics.mean(baseline_scores.values())
augmented_mean = statistics.mean(augmented_scores)

print("\n=== AGGREGATED RESULTS ===")
print(f"Baseline mean accuracy: {baseline_mean:.2%}")
print(f"Augmented mean accuracy: {augmented_mean:.2%}")
print(
    f"Mean improvement: "
    f"{100 * (augmented_mean - baseline_mean):+.2f} pp"
)

print(
    f"Augmented accuracy std: "
    f"{statistics.stdev(augmented_scores):.2%}"
)

print(
    f"Mean augmented flip consistency: "
    f"{statistics.mean(r['flip_consistency'] for r in augmentation_results):.2%}"
)

print(
    f"Seed-42 baseline flip consistency: 72.70%"
)


Map: 100%|██████████| 1000/1000 [00:00<00:00, 7581.78 examples/s]



AUGMENTED GRU | SEED 42


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 50.91% | Val: 53.60% | Val loss: 0.6872
Epoch 02 | Train: 61.92% | Val: 63.20% | Val loss: 0.6243
Epoch 03 | Train: 66.61% | Val: 65.70% | Val loss: 0.6148
Epoch 05 | Train: 69.40% | Val: 65.90% | Val loss: 0.6029
Epoch 06 | Train: 70.95% | Val: 66.00% | Val loss: 0.6092
Epoch 08 | Train: 73.90% | Val: 68.10% | Val loss: 0.6278
Epoch 10 | Train: 77.22% | Val: 65.90% | Val loss: 0.6524
Epoch 15 | Train: 88.05% | Val: 64.20% | Val loss: 0.8483
Epoch 20 | Train: 96.41% | Val: 64.80% | Val loss: 1.5351
Early stopping at epoch 23

=== AUGMENTED GRU RESULT ===
Seed: 42
Original accuracy: 68.10%
Swapped accuracy: 65.30%
Flip consistency: 72.80%
Correct in both orders: 53.10%
Best epoch: 8
Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
train/accuracy,▁▃▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇█████
train/loss,█▇▇▇▇▇▆▆▆▆▅▅▄▄▄▃▃▂▂▁▁▁▁
val/accuracy,▁▆▇▆▇▇▆█▇▇▇▇▆▇▆▇▆▇▆▆▇▆▆
val/loss,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▄▄▄▆▆▇██
best_epoch,8
best_val_accuracy,0.681
both_correct,0.531
epoch,23
epochs_trained,23
flip_consistency,0.728



AUGMENTED GRU | SEED 123


... (2 outputs left out)


AUGMENTED GRU | SEED 2026


Epoch 01 | Train: 50.88% | Val: 50.00% | Val loss: 0.6930
Epoch 02 | Train: 59.22% | Val: 64.20% | Val loss: 0.6293
Epoch 03 | Train: 65.97% | Val: 66.70% | Val loss: 0.6101
Epoch 05 | Train: 69.66% | Val: 66.80% | Val loss: 0.6058
Epoch 10 | Train: 77.65% | Val: 67.80% | Val loss: 0.6479
Epoch 15 | Train: 88.99% | Val: 64.80% | Val loss: 0.9697
Epoch 20 | Train: 96.36% | Val: 65.60% | Val loss: 1.6148
Epoch 25 | Train: 98.34% | Val: 65.40% | Val loss: 1.9366
Early stopping at epoch 25

=== AUGMENTED GRU RESULT ===
Seed: 2026
Original accuracy: 67.80%
Swapped accuracy: 67.40%
Flip consistency: 74.20%
Correct in both orders: 54.70%
Best epoch: 10
Checkpoint verification: PASS


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/accuracy,▁▂▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇███████
train/loss,██▇▇▇▆▆▆▆▅▅▅▄▄▃▃▂▃▂▂▁▁▁▁▁
val/accuracy,▁▇█▇█▇█▇▇███▇█▇█▇█▇▇▇▇▇▇▇
val/loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▄▅▆▆▇▇███
best_epoch,10
best_val_accuracy,0.678
both_correct,0.547
epoch,25
epochs_trained,25
flip_consistency,0.742



=== AUGMENTATION MULTI-SEED COMPARISON ===
Seed 42: Baseline=68.50% | Augmented=68.10% | Difference=-0.40 pp
Seed 123: Baseline=67.70% | Augmented=68.60% | Difference=+0.90 pp
Seed 2026: Baseline=67.90% | Augmented=67.80% | Difference=-0.10 pp

=== AGGREGATED RESULTS ===
Baseline mean accuracy: 68.03%
Augmented mean accuracy: 68.17%
Mean improvement: +0.13 pp
Augmented accuracy std: 0.40%
Mean augmented flip consistency: 72.73%
Seed-42 baseline flip consistency: 72.70%


### Augmentation Results

| Seed | Baseline accuracy | Augmented accuracy |
|---|---:|---:|
| 42 | 68.50% | 68.10% |
| 123 | 67.70% | 68.60% |
| 2026 | 67.90% | 67.80% |
| **Mean** | **68.03%** | **68.17%** |

The mean validation accuracy improved by only 0.13 percentage points.

Mean prediction-flip consistency was 72.73%, almost identical to the original seed-42 diagnostic (72.70%).

**Conclusion:** Hypothesis-swapping augmentation did not provide a meaningful improvement in accuracy or answer-order consistency.